# 🚦 FedMARL-ATSC · Notebook 02
## Tích hợp CityFlow ↔ Federated Learning · Data Pipeline Demo

> **Mục tiêu**: Chứng minh luồng dữ liệu (Data Pipeline) chạy xuyên suốt từ **CityFlow** (simulator thật) qua **NodeAgent** (mạng Nơ-ron phân rã) đến **FederatedServer** (tổng hợp FedAvg) và trả về — không cần RL hội tụ.

---

### Tổng quan Luồng Dữ liệu

```
CityFlow Engine (sample_map_2)
  │
  │  get_lane_waiting_vehicle_count()
  │  get_vehicle_speed()
  ▼
TrafficEnv.get_state()           → obs: np.ndarray (84,) × 2 giao lộ
  │
  │  [queue_norm, speed_norm, density_norm] × 28 lanes/giao lộ
  ▼
NodeAgent.select_action(obs)     → action: int ∈ {0,1,2,3}
  │                                (ε-greedy qua Q-network)
  ▼
TrafficEnv.step(actions)         → obs', reward, terminated, truncated, info
  │  (CityFlow: apply yellow → green → next_step × 30)
  │
  ├── NodeAgent.compute_dummy_loss(obs')  → loss.backward()
  │
  └── [Mỗi N bước] ─────────────────────────────────────────────────────
        │
        ▼
      NodeAgent.get_shared_weights()      → backbone state_dict (không có head)
        │
        ▼
      FederatedServer.aggregate([w1,w2])  → global_weights (FedAvg)
        │
        ▼
      FederatedServer.broadcast()         → NodeAgent.set_shared_weights()
                                            (chỉ backbone, head giữ nguyên)
```

---

### Bản đồ Mô phỏng: `sample_map_2`

| Thuộc tính | Giá trị |
|---|---|
| Giao lộ có đèn | 2: `intersection_1_1`, `intersection_2_1` |
| Làn xe vào / giao lộ | 28 (4 hướng × 7 làn) |
| Pha xanh / giao lộ | 4 (index CityFlow: 0, 2, 4, 6) |
| obs_dim / giao lộ | 84 (28 × 3 features) |
| obs_dim tổng hợp | 168 |


---
## Cell 1 · Setup: Imports & Paths

In [1]:
import sys, os, copy, logging, json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Thêm thư mục gốc vào sys.path để import các module dự án ──────────────
WORKSPACE = Path("__file__").resolve().parent.parent   # notebooks/ → ĐACN/
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

# ── Logging ───────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.WARNING,
    format="%(levelname)s [%(name)s] %(message)s"
)
# Hiển thị INFO từ module dự án
for mod in ["env.cityflow_wrapper", "agents.node_agent", "federated.server"]:
    logging.getLogger(mod).setLevel(logging.INFO)

# ── Import các lớp dự án ──────────────────────────────────────────────────
from env.cityflow_wrapper import TrafficEnv
from agents.node_agent    import NodeAgent
from federated.server     import FederatedServer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
print("Imports OK ✓")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/ipyk

Device  : cpu
PyTorch : 2.1.1
Imports OK ✓


---
## Cell 2 · Hyperparameters

In [2]:
# ── Đường dẫn CityFlow ────────────────────────────────────────────────────
CITYFLOW_DIR    = WORKSPACE / "cityflow"
CITYFLOW_CONFIG = CITYFLOW_DIR / "build/generator_cityflow/sample_map_2/config.json"

assert CITYFLOW_CONFIG.exists(), f"Không tìm thấy: {CITYFLOW_CONFIG}"
print(f"Config CityFlow : {CITYFLOW_CONFIG}")

# ── Vòng lặp mô phỏng ────────────────────────────────────────────────────
TOTAL_SIM_STEPS  = 50    # Tổng số action-steps (mỗi step = yellow+green ≈ 33s)
FL_INTERVAL      = 10    # Tiến hành FL sau mỗi N steps
LOCAL_TRAIN_ITERS = 3    # Số lần backward() cục bộ trước mỗi vòng FL

# ── Môi trường ───────────────────────────────────────────────────────────
GREEN_DURATION  = 10     # Giây mỗi pha xanh (giảm để demo nhanh)
YELLOW_DURATION = 3      # Giây đèn vàng bắt buộc (C-03)

# ── Mạng Nơ-ron NodeAgent ────────────────────────────────────────────────
# Các tham số này sẽ được tự động lấy từ roadnet (obs_dim, num_phases)
HIDDEN_DIM   = 128
BACKBONE_OUT = 64
LR           = 1e-3
EPSILON      = 0.3   # Khám phá cao hơn khi chưa train thật

# ── Federated Learning ───────────────────────────────────────────────────
FL_ALGORITHM     = "fedavg"
USE_CLUSTERING   = False  # Phase 1: tắt clustering, bật ở Phase 2

print(f"Tổng simulation steps : {TOTAL_SIM_STEPS}")
print(f"FL interval           : mỗi {FL_INTERVAL} steps")
print(f"Số vòng FL dự kiến    : {TOTAL_SIM_STEPS // FL_INTERVAL}")
print(f"Green / Yellow        : {GREEN_DURATION}s / {YELLOW_DURATION}s")

AssertionError: Không tìm thấy: /Users/buiquanghuy/Documents/cityflow/build/generator_cityflow/sample_map_2/config.json

---
## Cell 3 · Khởi tạo TrafficEnv

Lớp `TrafficEnv` tự động:
- Đọc `roadnet.json` → phát hiện 2 giao lộ có đèn, 28 làn xe vào, 4 pha xanh
- Vá `config.json` để bật `rlTrafficLight=true` (không sửa file gốc)
- Tính `obs_dim = 28 × 3 = 84` cho mỗi giao lộ

In [ ]:
env = TrafficEnv(
    config_path     = str(CITYFLOW_CONFIG),
    green_duration  = GREEN_DURATION,
    yellow_duration = YELLOW_DURATION,
    min_green_steps = 1,
    reward_weights  = {"queue": 0.4, "pressure": 0.6, "phase_change": 0.05},
)

print(env)
print()
print("=" * 60)
print("THÔNG TIN GIAO LỘ (đọc từ roadnet.json)")
print("=" * 60)
for ix_id, ix_info in env.intersections.items():
    print(f"\n  {ix_id}:")
    print(f"    obs_dim          : {ix_info.obs_dim}")
    print(f"    num_green_phases : {ix_info.num_green_phases}")
    print(f"    green_phase_idx  : {ix_info.green_phase_indices}  (pha CityFlow)")
    print(f"    incoming roads   : {ix_info.incoming_roads}")
    print(f"    incoming lanes   : {len(ix_info.incoming_lane_ids)} lanes")

print(f"\n  obs_dim TỔNG HỢP  : {env.obs_dim}  (= {len(env.intersections)} × 84)")

---
## Cell 4 · Khởi tạo NodeAgent

Mỗi giao lộ nhận một NodeAgent riêng. `obs_dim` và `num_phases` được lấy
**trực tiếp từ metadata của TrafficEnv** — đảm bảo khớp chính xác.

In [ ]:
agents = {}

for ix_id, ix_info in env.intersections.items():
    agents[ix_id] = NodeAgent(
        intersection_id = ix_id,
        obs_dim         = ix_info.obs_dim,          # 84 — từ roadnet thật
        num_phases      = ix_info.num_green_phases,  # 4  — từ roadnet thật
        hidden_dim      = HIDDEN_DIM,
        backbone_out    = BACKBONE_OUT,
        lr              = LR,
        epsilon         = EPSILON,
        device          = DEVICE,
    )
    print(agents[ix_id])
    print(f"  TrafficAgentNet:")
    total_params = sum(p.numel() for p in agents[ix_id].net.parameters())
    backbone_params = sum(p.numel() for p in agents[ix_id].net.shared_feature_extractor.parameters())
    head_params     = sum(p.numel() for p in agents[ix_id].net.local_policy_head.parameters())
    print(f"    Backbone params (FL) : {backbone_params:,}")
    print(f"    Head params (private): {head_params:,}")
    print(f"    Total params         : {total_params:,}")
    print()

print(f"Đã tạo {len(agents)} NodeAgent ✓")

---
## Cell 5 · Khởi tạo FederatedServer

In [ ]:
server = FederatedServer(
    clients        = list(agents.values()),
    use_clustering = USE_CLUSTERING,
    num_clusters   = 2,
    fedprox_mu     = 0.0,
)
print(server)

# ── Kiểm tra get_shared_weights() trước khi vòng lặp bắt đầu ─────────────
print("\nKiểm tra get_shared_weights() (R-02):")
for ix_id, agent in agents.items():
    sw = agent.get_shared_weights()
    all_keys    = set(agent.net.state_dict().keys())
    shared_keys = set(sw.keys())
    head_keys   = {k for k in all_keys if k not in {f"shared_feature_extractor.{k2}" for k2 in sw}}

    # R-02: Không được có key của local_policy_head trong shared weights
    head_leaked = any("local_policy_head" in k for k in sw.keys())
    print(f"  [{ix_id}] shared keys={len(sw)} | head_leaked={head_leaked} → {'✓ OK' if not head_leaked else '✗ VI PHẠM R-02!'}")

---
## Cell 6 · Reset & Warm-up

Gọi `env.reset()` để CityFlow Engine tải xe lên đường và trả về observation đầu tiên.

In [ ]:
obs, info = env.reset(seed=42)

print(f"Observation shape  : {obs.shape}         (= obs_dim_total={env.obs_dim})")
print(f"Obs min/max/mean   : {obs.min():.3f} / {obs.max():.3f} / {obs.mean():.3f}")
print(f"Vehicles on road   : {info['vehicle_count']}")
print(f"Total waiting      : {info.get('total_waiting', 0)}")
print(f"Sim time           : {info.get('current_time', 0)} s")
print()

# Hiển thị obs từng giao lộ
print("Observation từng giao lộ (10 feature đầu):")
offset = 0
for ix_id, ix_info in env.intersections.items():
    ix_obs = obs[offset: offset + ix_info.obs_dim]
    print(f"  [{ix_id}] obs[:10] = {ix_obs[:10].round(3)}")
    offset += ix_info.obs_dim

---
## Cell 7 · Simulation Loop Chính

```
for step in range(TOTAL_SIM_STEPS):
    a. Mỗi Agent lấy obs cục bộ → forward Q-network → chọn action
    b. TrafficEnv.step(actions) → CityFlow chạy yellow+green
    c. compute_dummy_loss(obs') → backward() → optimizer.step()
    d. [Mỗi FL_INTERVAL steps]
       get_shared_weights() → server.aggregate() → server.broadcast()
```

In [ ]:
# ── Bộ ghi log ───────────────────────────────────────────────────────────
log = {
    "step":          [],
    "reward":        [],
    "total_waiting": [],
    "vehicle_count": [],
    "fl_rounds":     [],   # step index của mỗi vòng FL
    "loss": {ix_id: [] for ix_id in env.intersection_ids},
}

SEP = "═" * 68
print(SEP)
print(f"  FedMARL-ATSC · CityFlow + FL Integration Demo")
print(f"  Map: sample_map_2  |  Steps: {TOTAL_SIM_STEPS}  |  FL interval: {FL_INTERVAL}")
print(SEP)

current_obs = obs.copy()

for step in range(1, TOTAL_SIM_STEPS + 1):

    # ──────────────────────────────────────────────────────────────────────
    # (a) Mỗi Agent lấy obs cục bộ → chọn action
    # ──────────────────────────────────────────────────────────────────────
    actions = {}
    offset  = 0
    for ix_id, ix_info in env.intersections.items():
        ix_obs          = current_obs[offset: offset + ix_info.obs_dim]
        actions[ix_id]  = agents[ix_id].select_action(ix_obs)
        offset         += ix_info.obs_dim

    # ──────────────────────────────────────────────────────────────────────
    # (b) TrafficEnv.step() → CityFlow: yellow phase + green phase
    # ──────────────────────────────────────────────────────────────────────
    next_obs, reward, terminated, truncated, info = env.step(actions)

    # ──────────────────────────────────────────────────────────────────────
    # (c) Local training — Dummy Loss (backward() thật, loss function giả)
    # ──────────────────────────────────────────────────────────────────────
    step_losses = {}
    offset = 0
    for ix_id, ix_info in env.intersections.items():
        ix_next_obs = next_obs[offset: offset + ix_info.obs_dim]
        # LOCAL_TRAIN_ITERS lần backward() mỗi step
        iter_losses = [
            agents[ix_id].compute_dummy_loss(ix_next_obs)
            for _ in range(LOCAL_TRAIN_ITERS)
        ]
        step_losses[ix_id] = float(np.mean(iter_losses))
        log["loss"][ix_id].append(step_losses[ix_id])
        offset += ix_info.obs_dim

    # ──────────────────────────────────────────────────────────────────────
    # Ghi log step
    # ──────────────────────────────────────────────────────────────────────
    log["step"].append(step)
    log["reward"].append(reward)
    log["total_waiting"].append(info.get("total_waiting", 0))
    log["vehicle_count"].append(info.get("vehicle_count", 0))

    if step % 5 == 0 or step == 1:
        loss_str = "  ".join(
            f"{ix_id.split('_',1)[1]}: {v:.4f}"
            for ix_id, v in step_losses.items()
        )
        print(
            f"  Step {step:3d}/{TOTAL_SIM_STEPS}"
            f"  reward={reward:7.2f}"
            f"  waiting={info.get('total_waiting',0):4d}"
            f"  veh={info.get('vehicle_count',0):4d}"
            f"  loss [{loss_str}]"
        )

    # ──────────────────────────────────────────────────────────────────────
    # (d) Federated Learning — mỗi FL_INTERVAL steps
    # ──────────────────────────────────────────────────────────────────────
    if step % FL_INTERVAL == 0:
        fl_log = server.run_round(algorithm=FL_ALGORITHM)
        log["fl_rounds"].append(step)
        print(f"\n  {'─'*62}")
        print(
            f"  🔄 FL Round {fl_log['fl_round']}"
            f"  |  {FL_ALGORITHM.upper()}"
            f"  |  {fl_log['num_clients']} clients"
            f"  |  {fl_log['param_tensors']} tensors aggregated"
        )
        # Xác nhận R-02: backbone đồng bộ
        ref = list(agents.values())[0].get_shared_weights()
        synced = all(
            all(torch.allclose(ref[k], a.get_shared_weights()[k]) for k in ref)
            for a in list(agents.values())[1:]
        )
        print(f"  ✓ Backbone đồng bộ : {synced}  |  Head riêng tư: ✓ (không chia sẻ)")
        print(f"  {'─'*62}\n")

    current_obs = next_obs

    if terminated or truncated:
        print(f"\nEpisode kết thúc sớm tại step {step}.")
        break

print(f"\n{SEP}")
print(f"  ✅ Simulation hoàn thành! {step} steps, {len(log['fl_rounds'])} vòng FL.")
print(SEP)

env.close()

---
## Cell 8 · Trực quan hoá Kết quả

In [ ]:
steps = log["step"]

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle(
    "FedMARL-ATSC · CityFlow + Federated Learning Integration Demo\n"
    f"Map: sample_map_2 | {len(env.intersections)} giao lộ | {TOTAL_SIM_STEPS} steps",
    fontsize=13, fontweight="bold"
)

# Vẽ đường dọc tại mỗi vòng FL
def _add_fl_lines(ax):
    for i, fl_step in enumerate(log["fl_rounds"]):
        ax.axvline(
            x=fl_step, color="purple", linestyle="--", linewidth=0.9, alpha=0.6,
            label="FL round" if i == 0 else None
        )

colors = ["#2196F3", "#FF5722"]

# ── (1) Reward ──────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(steps, log["reward"], color="#4CAF50", linewidth=1.8, label="Reward")
_add_fl_lines(ax)
ax.set_xlabel("Simulation Step")
ax.set_ylabel("Reward")
ax.set_title("Reward theo thời gian")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── (2) Total Waiting Vehicles ───────────────────────────────────────────
ax = axes[0, 1]
ax.plot(steps, log["total_waiting"], color="#FF5722", linewidth=1.8, label="Total Waiting")
ax.fill_between(steps, log["total_waiting"], alpha=0.15, color="#FF5722")
_add_fl_lines(ax)
ax.set_xlabel("Simulation Step")
ax.set_ylabel("Số xe chờ")
ax.set_title("Tổng số xe đang chờ (Hàng đợi)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── (3) Loss từng NodeAgent ───────────────────────────────────────────────
ax = axes[1, 0]
for i, (ix_id, loss_vals) in enumerate(log["loss"].items()):
    ax.plot(steps, loss_vals, color=colors[i], linewidth=1.5,
            label=ix_id, linestyle=["-","--"][i])
_add_fl_lines(ax)
ax.set_xlabel("Simulation Step")
ax.set_ylabel("Dummy MSE Loss")
ax.set_title("Loss Cục bộ từng NodeAgent\n(Dummy Loss — RL trainer sẽ thay thế)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── (4) Vehicle Count ─────────────────────────────────────────────────────
ax = axes[1, 1]
ax.plot(steps, log["vehicle_count"], color="#9C27B0", linewidth=1.8, label="Vehicles")
_add_fl_lines(ax)
ax.set_xlabel("Simulation Step")
ax.set_ylabel("Số xe trên mạng")
ax.set_title("Tổng số xe trong mô phỏng")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(WORKSPACE / "data/02_integration_demo_results.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Plot đã lưu → data/02_integration_demo_results.png")

---
## Cell 9 · Kiểm tra Toàn diện Luồng Dữ liệu

Xác nhận các bất biến kỹ thuật theo `PROJECT_GUIDELINES.md`:

In [ ]:
print("=" * 68)
print(" KIỂM TRA TOÀN DIỆN LUỒNG DỮ LIỆU (PROJECT_GUIDELINES.md)")
print("=" * 68)

agent_list = list(agents.values())

# ── R-02: get_shared_weights() CHỈ trả về backbone ───────────────────────
print("\n[R-02] Kiểm tra get_shared_weights() chỉ chứa backbone keys:")
all_pass = True
for ix_id, agent in agents.items():
    sw = agent.get_shared_weights()
    head_leaked = any("local_policy_head" in k for k in sw.keys())
    backbone_missing = not any("shared_feature_extractor" in k for k in sw.keys())
    ok = not head_leaked and not backbone_missing
    all_pass = all_pass and ok
    print(f"  [{ix_id}] head_leaked={head_leaked} | backbone_ok={not backbone_missing} → {'✅ PASS' if ok else '❌ FAIL'}")
print(f"  R-02 Overall: {'✅ PASS' if all_pass else '❌ FAIL'}")

# ── Backbone đồng bộ sau broadcast ───────────────────────────────────────
print("\n[FL] Backbone đồng bộ sau vòng FL cuối:")
if len(agent_list) >= 2:
    ref = agent_list[0].get_shared_weights()
    backbone_synced = all(
        all(torch.allclose(ref[k], a.get_shared_weights()[k]) for k in ref)
        for a in agent_list[1:]
    )
    print(f"  Backbone đồng bộ : {backbone_synced} → {'✅ PASS' if backbone_synced else '❌ FAIL'}")

# ── R-03: Head riêng tư, khác nhau giữa agents ───────────────────────────
print("\n[R-03] Policy head KHÁC NHAU giữa các agents (personalisation):")
if len(agent_list) >= 2:
    ref_head   = agent_list[0].net.local_policy_head.state_dict()
    heads_differ = any(
        not all(torch.allclose(ref_head[k], a.net.local_policy_head.state_dict()[k]) for k in ref_head)
        for a in agent_list[1:]
    )
    print(f"  Heads khác nhau  : {heads_differ} → {'✅ PASS (personalised)' if heads_differ else '⚠ SAME (cùng random init)'}")

# ── Tổng kết vòng FL ─────────────────────────────────────────────────────
print("\n[FL] Tóm tắt các vòng Federated Learning:")
for fl_log in server.round_logs:
    print(
        f"  Vòng {fl_log['fl_round']:2d}"
        f" | {fl_log['algorithm'].upper()}"
        f" | {fl_log['num_clients']} clients"
        f" | {fl_log['param_tensors']} tensors"
    )

print()
print("=" * 68)
print(" KẾT LUẬN")
print("=" * 68)
print("""
 ✅ TrafficEnv đọc CityFlow roadnet thật & trả về obs chuẩn hoá
 ✅ NodeAgent forward obs qua Q-network phân rã → action hợp lệ
 ✅ TrafficEnv.step() thực thi ràng buộc an toàn (yellow, min_green)
 ✅ Dummy loss.backward() hoạt động, gradients chạy qua toàn bộ network
 ✅ get_shared_weights() chỉ trả về backbone (R-02)
 ✅ FederatedServer.aggregate() tổng hợp FedAvg thành công
 ✅ FederatedServer.broadcast() đồng bộ backbone, head giữ nguyên (R-03)

 → Luồng dữ liệu (Pipeline) CityFlow → FedMARL → Server HOẠT ĐỘNG! 🎉
 → Thay compute_dummy_loss() bằng DQN TD-error để bắt đầu train thật.
""")

---
## Cell 10 · Bước Tiếp theo (Phase 2)

| Thành phần | Hiện tại | Phase 2 |
|---|---|---|
| `TrafficEnv` | ✅ Đầy đủ, ràng buộc an toàn | Thêm neighbor embedding |
| `NodeAgent.train()` | Dummy MSE loss | ➡️ DQN TD-error / PPO gradient |
| `ReplayBuffer` | Chưa có | ➡️ `agents/replay_buffer.py` |
| `FederatedServer` | FedAvg ✅ | ➡️ Bật `use_clustering=True` |
| Map | sample_map_2 (2 giao lộ) | ➡️ Grid 3×3, Arterial |
| Experiment tracking | Chưa | ➡️ `wandb.init()` |

```python
# Thay thế compute_dummy_loss() bằng:
from agents.dqn import DQNTrainer
trainer = DQNTrainer(agent=agents[ix_id], replay_buffer=buffer)
loss = trainer.update()          # TD-error thật
```
